# REM Experiment Notebook (App-Exact Logic)

Ce notebook reproduit la logique REM de l'app le plus fidèlement possible pour comparer différents réglages de smoothing/threshold.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy.signal import butter, filtfilt, iirnotch

warnings.filterwarnings("ignore", category=RuntimeWarning)


In [ ]:
# ---- Paths / IDs ----
# Motion PID can differ from EEG PID (e.g. motion uses Soso, EEG file is EEG_Soso3.csv)
PID_MOTION = "Soso"
PID_EEG = "Soso3"
RAW_MOTION_CSV = Path("/Users/solenenoize/Desktop/LuciDreams/plot/data_night/motion_per_second_series.csv")
EEG_CSV = Path(f"/Users/solenenoize/Desktop/LuciDreams/plot/EEG_{PID_EEG}.csv")

# ---- App-like constants ----
REM_GATE_SECONDS = 4 * 60 * 60
REM_EXIT_HOLD_SEC = 10

# Default run config (you can override per experiment cell)
SMOOTH_WINDOW_SEC = 120
REM_STILLNESS_PERCENT = 0.80
PER_REM_PRE_SEC = 60
PER_REM_POST_SEC = 60

# EEG plotting config (inline only; no file saving)
EEG_BIN_SEC = 0.5
BANDPASS_LOW_HZ = 0.5
BANDPASS_HIGH_HZ = 40.0
NOTCH_HZ = 60.0
NOTCH_Q = 30.0
PER_REM_LOWPASS_HZ = 40.0


In [ ]:
def choose_session(df_motion: pd.DataFrame, pid: str) -> pd.DataFrame:
    d = df_motion.copy()
    d["pid"] = d["pid"].astype(str)
    out = d[d["pid"] == str(pid)].copy()
    if out.empty:
        available = sorted(d["pid"].dropna().astype(str).unique().tolist())
        raise ValueError(f"No rows for pid {pid}. Available pids: {available[:15]}")

    out["second_index"] = pd.to_numeric(out["second_index"], errors="coerce")
    out["motion_per_second"] = pd.to_numeric(out["motion_per_second"], errors="coerce")
    out = out.dropna(subset=["second_index", "motion_per_second"]).copy()
    out["second_index"] = out["second_index"].astype(int)
    out = out.sort_values("second_index").reset_index(drop=True)

    sess_col = "session_start_boston" if "session_start_boston" in out.columns else "session_start"
    if sess_col not in out.columns:
        raise ValueError("Missing session start column in motion CSV")

    # Keep one night: choose the session with most rows for this pid.
    out["_sess"] = out[sess_col].astype(str)
    best_session = out.groupby("_sess").size().sort_values(ascending=False).index[0]
    out = out[out["_sess"] == best_session].copy()
    out["session_start_boston"] = pd.to_datetime(out[sess_col], errors="coerce")
    out = out.dropna(subset=["session_start_boston"]).reset_index(drop=True)
    out["event_time_boston"] = out["session_start_boston"] + pd.to_timedelta(out["second_index"], unit="s")
    return out


def compute_smoothed_like_app(raw_series: pd.Series, window_sec: int) -> pd.Series:
    """
    Exact app-like smoothing:
    - Append each raw value into a rolling buffer
    - Keep only last `window_sec` values
    - Emit NaN until buffer is full
    - Then emit rolling mean
    """
    vals = pd.to_numeric(raw_series, errors="coerce").astype(float).tolist()
    out = []
    buf = []
    rolling_sum = 0.0

    for v in vals:
        if not np.isfinite(v):
            out.append(np.nan)
            continue
        buf.append(v)
        rolling_sum += v
        if len(buf) > window_sec:
            rolling_sum -= buf.pop(0)

        if len(buf) >= window_sec:
            out.append(rolling_sum / len(buf))
        else:
            out.append(np.nan)

    return pd.Series(out, index=raw_series.index, dtype=float)


def is_still_smoothed(smoothed_history, current_smoothed, pct=0.8):
    values = [v for v in smoothed_history if np.isfinite(v)]
    if len(values) == 0 or not np.isfinite(current_smoothed):
        return False
    below_count = sum(1 for v in values if float(current_smoothed) < float(v))
    return (below_count / len(values)) >= float(pct)


def simulate_rem_episodes_exact_app(df: pd.DataFrame, stillness_pct: float, rem_gate_seconds: int, rem_exit_hold_sec: int) -> pd.DataFrame:
    """
    Mirrors app REM state-machine (without train/cue side-effects):
    - secondsRecorded = second_index + 1 (1-based)
    - smoothed history grows whenever current smoothed exists
    - afterGate required before REM logic runs
    - immediate entry when stillness is true
    - exit after `rem_exit_hold_sec` consecutive non-still seconds
    - duration_sec = end_sec - start_sec
    """
    in_rem = False
    consecutive_non_still = 0
    rem_start_sec = None
    episodes = []
    smoothed_history = []

    for _, row in df.iterrows():
        sec_idx = int(row["second_index"])
        sec_rec = sec_idx + 1
        sm = float(row["motion_smoothed_exp"]) if np.isfinite(row["motion_smoothed_exp"]) else np.nan

        if np.isfinite(sm):
            smoothed_history.append(sm)

        after_gate = (sec_rec > rem_gate_seconds) and (len(smoothed_history) > 0) and np.isfinite(sm)
        if not after_gate:
            in_rem = False
            consecutive_non_still = 0
            rem_start_sec = None
            continue

        still = is_still_smoothed(smoothed_history, sm, stillness_pct)

        if not in_rem:
            if still:
                rem_start_sec = sec_rec
                in_rem = True
                consecutive_non_still = 0
        else:
            if still:
                consecutive_non_still = 0
            else:
                consecutive_non_still += 1
                if consecutive_non_still >= rem_exit_hold_sec:
                    end_sec = sec_rec
                    episodes.append({
                        "episode_index": len(episodes) + 1,
                        "start_sec": int(rem_start_sec),
                        "end_sec": int(end_sec),
                        "duration_sec": int(end_sec - rem_start_sec),
                    })
                    in_rem = False
                    consecutive_non_still = 0
                    rem_start_sec = None

    if in_rem and rem_start_sec is not None:
        end_sec = int(df["second_index"].max()) + 1
        episodes.append({
            "episode_index": len(episodes) + 1,
            "start_sec": int(rem_start_sec),
            "end_sec": int(end_sec),
            "duration_sec": int(end_sec - rem_start_sec),
        })

    rem_df = pd.DataFrame(episodes)
    if rem_df.empty:
        return rem_df

    start0 = pd.to_datetime(df["session_start_boston"].iloc[0])
    rem_df["session_start_boston"] = start0
    rem_df["episode_start_boston"] = start0 + pd.to_timedelta(rem_df["start_sec"], unit="s")
    rem_df["episode_end_boston"] = start0 + pd.to_timedelta(rem_df["end_sec"], unit="s")
    return rem_df


In [ ]:
def robust_zscore(series: pd.Series) -> pd.Series:
    x = pd.to_numeric(series, errors="coerce").astype(float)
    med = np.nanmedian(x)
    mad = np.nanmedian(np.abs(x - med))
    if not np.isfinite(mad) or mad < 1e-12:
        std = np.nanstd(x)
        if not np.isfinite(std) or std < 1e-12:
            return x * 0.0
        return (x - med) / std
    return 0.6745 * (x - med) / mad


def estimate_sampling_hz(ts: pd.Series) -> float:
    t = pd.to_datetime(ts, errors="coerce").dropna().sort_values()
    if len(t) < 3:
        return np.nan
    dt = t.diff().dropna().dt.total_seconds().values
    dt = dt[np.isfinite(dt) & (dt > 0)]
    if len(dt) == 0:
        return np.nan
    return 1.0 / np.median(dt)


def bandpass_notch_filter(x: np.ndarray, fs: float, low_hz: float, high_hz: float, notch_hz: float, notch_q: float) -> np.ndarray:
    y = np.asarray(x, dtype=float)
    if not np.isfinite(fs) or fs <= 0:
        return y
    nyq = 0.5 * fs

    if 0 < low_hz < high_hz < nyq:
        b, a = butter(4, [low_hz / nyq, high_hz / nyq], btype="band")
        y = filtfilt(b, a, y)

    if 0 < notch_hz < nyq:
        b, a = iirnotch(notch_hz / nyq, notch_q)
        y = filtfilt(b, a, y)

    return y


def lowpass_filter(x: np.ndarray, fs: float, cutoff_hz: float) -> np.ndarray:
    y = np.asarray(x, dtype=float)
    if not np.isfinite(fs) or fs <= 0:
        return y
    nyq = 0.5 * fs
    if not (0 < cutoff_hz < nyq):
        return y
    b, a = butter(4, cutoff_hz / nyq, btype="low")
    return filtfilt(b, a, y)


def load_and_prepare_eeg(eeg_csv: Path) -> pd.DataFrame:
    if not eeg_csv.exists():
        raise FileNotFoundError(f"EEG file not found: {eeg_csv}")

    eeg = pd.read_csv(eeg_csv)
    required = {"Timestamp", "RAW_AF7", "RAW_AF8"}
    missing = [c for c in required if c not in eeg.columns]
    if missing:
        raise ValueError(f"Missing EEG columns: {missing}")

    eeg = eeg.copy()
    eeg["timestamp_boston"] = pd.to_datetime(eeg["Timestamp"], errors="coerce")
    eeg["RAW_AF7"] = pd.to_numeric(eeg["RAW_AF7"], errors="coerce")
    eeg["RAW_AF8"] = pd.to_numeric(eeg["RAW_AF8"], errors="coerce")
    eeg = eeg.dropna(subset=["timestamp_boston", "RAW_AF7", "RAW_AF8"]).sort_values("timestamp_boston").reset_index(drop=True)

    fs = estimate_sampling_hz(eeg["timestamp_boston"])
    if np.isfinite(fs) and fs > 1:
        eeg["RAW_AF7"] = bandpass_notch_filter(eeg["RAW_AF7"].values, fs, BANDPASS_LOW_HZ, BANDPASS_HIGH_HZ, NOTCH_HZ, NOTCH_Q)
        eeg["RAW_AF8"] = bandpass_notch_filter(eeg["RAW_AF8"].values, fs, BANDPASS_LOW_HZ, BANDPASS_HIGH_HZ, NOTCH_HZ, NOTCH_Q)

    eeg["af7_z"] = robust_zscore(eeg["RAW_AF7"])
    eeg["af8_z"] = robust_zscore(eeg["RAW_AF8"])

    # Aggregate to fixed bins for readability.
    t0 = eeg["timestamp_boston"].iloc[0]
    sec = (eeg["timestamp_boston"] - t0).dt.total_seconds()
    bin_idx = np.floor(sec / EEG_BIN_SEC).astype(int)
    eeg["_bin"] = bin_idx

    agg = eeg.groupby("_bin", as_index=False).agg(
        timestamp_boston=("timestamp_boston", "min"),
        af7_z=("af7_z", "mean"),
        af8_z=("af8_z", "mean"),
    )

    fs_agg = 1.0 / EEG_BIN_SEC
    agg["af7_z"] = lowpass_filter(agg["af7_z"].values, fs_agg, PER_REM_LOWPASS_HZ)
    agg["af8_z"] = lowpass_filter(agg["af8_z"].values, fs_agg, PER_REM_LOWPASS_HZ)
    return agg


In [ ]:
def plot_motion_per_rem(df: pd.DataFrame, rem_df: pd.DataFrame, title_prefix: str, pre_sec: int = 60, post_sec: int = 60):
    if rem_df.empty:
        print("No simulated REM episodes with current settings")
        return

    for _, ep in rem_df.iterrows():
        eidx = int(ep["episode_index"])
        t0 = pd.to_datetime(ep["episode_start_boston"]) - pd.to_timedelta(pre_sec, unit="s")
        t1 = pd.to_datetime(ep["episode_end_boston"]) + pd.to_timedelta(post_sec, unit="s")

        d = df[(df["event_time_boston"] >= t0) & (df["event_time_boston"] <= t1)].copy()
        if d.empty:
            continue

        fig, ax = plt.subplots(figsize=(12, 4))
        ax.plot(d["event_time_boston"], d["motion_smoothed_exp"], color="tab:blue", lw=1.5, label="smoothed motion")
        ax.axvline(pd.to_datetime(ep["episode_start_boston"]), color="yellow", lw=2, label="REM start")
        ax.axvline(pd.to_datetime(ep["episode_end_boston"]), color="black", lw=2, label="REM end")

        ax.set_title(f"{title_prefix} | REM {eidx}")
        ax.set_xlabel("Boston time")
        ax.set_ylabel("Motion (smoothed)")
        ax.grid(alpha=0.2)
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M:%S"))
        ax.legend(loc="upper right")
        fig.autofmt_xdate()
        plt.show()


def plot_eeg_per_rem(eeg_df: pd.DataFrame, rem_df: pd.DataFrame, title_prefix: str, pre_sec: int = 60, post_sec: int = 60):
    if rem_df.empty:
        print("No simulated REM episodes with current settings")
        return

    for _, ep in rem_df.iterrows():
        eidx = int(ep["episode_index"])
        rem_start = pd.to_datetime(ep["episode_start_boston"])
        rem_end = pd.to_datetime(ep["episode_end_boston"])
        t0 = rem_start - pd.to_timedelta(pre_sec, unit="s")
        t1 = rem_end + pd.to_timedelta(post_sec, unit="s")

        d = eeg_df[(eeg_df["timestamp_boston"] >= t0) & (eeg_df["timestamp_boston"] <= t1)].copy()
        if d.empty:
            continue

        rel_sec = (d["timestamp_boston"] - rem_start).dt.total_seconds()

        fig, ax = plt.subplots(figsize=(12, 4))
        ax.plot(rel_sec, d["af7_z"], lw=1.0, color="tab:purple", label="AF7 (z)")
        ax.plot(rel_sec, d["af8_z"], lw=1.0, color="tab:orange", label="AF8 (z)")
        ax.axvline(0, color="yellow", lw=2, label="REM start")
        ax.axvline((rem_end - rem_start).total_seconds(), color="black", lw=2, label="REM end")

        ax.set_title(f"{title_prefix} | EEG around REM {eidx}")
        ax.set_xlabel("Relative time from REM start (s)")
        ax.set_ylabel("Filtered normalized EEG")
        ax.grid(alpha=0.2)
        ax.legend(loc="upper right")
        plt.show()


In [ ]:
# Load motion and prepare current night
motion_all = pd.read_csv(RAW_MOTION_CSV)
motion_df = choose_session(motion_all, PID_MOTION)
print(f"Motion PID={PID_MOTION} | rows={len(motion_df)} | session_start={motion_df['session_start_boston'].iloc[0]}")

# Load EEG once
eeg_df = load_and_prepare_eeg(EEG_CSV)
print(f"EEG PID={PID_EEG} | file={EEG_CSV.name} | rows after preprocessing={len(eeg_df)}")


In [ ]:
def run_experiment(window_sec: int, stillness_pct: float, label: str):
    d = motion_df.copy()
    d["motion_smoothed_exp"] = compute_smoothed_like_app(d["motion_per_second"], window_sec)

    rem_df = simulate_rem_episodes_exact_app(
        d,
        stillness_pct=stillness_pct,
        rem_gate_seconds=REM_GATE_SECONDS,
        rem_exit_hold_sec=REM_EXIT_HOLD_SEC,
    )

    print(f"\n[{label}] window={window_sec}s | stillness={stillness_pct:.2f} | episodes={len(rem_df)}")
    if not rem_df.empty:
        display(rem_df[["episode_index", "start_sec", "end_sec", "duration_sec", "episode_start_boston", "episode_end_boston"]])

    plot_motion_per_rem(d, rem_df, title_prefix=label, pre_sec=PER_REM_PRE_SEC, post_sec=PER_REM_POST_SEC)
    plot_eeg_per_rem(eeg_df, rem_df, title_prefix=label, pre_sec=PER_REM_PRE_SEC, post_sec=PER_REM_POST_SEC)
    return d, rem_df


## Experiment 1: 2 min window, 80% stillness

In [ ]:
exp1_df, exp1_rem = run_experiment(
    window_sec=120,
    stillness_pct=0.80,
    label="2min / 80%"
)


## Experiment 2: 5 min window, 80% stillness

In [ ]:
exp2_df, exp2_rem = run_experiment(
    window_sec=300,
    stillness_pct=0.80,
    label="5min / 80%"
)


## Experiment 3: 3 min window, 80% stillness

In [ ]:
exp3_df, exp3_rem = run_experiment(
    window_sec=180,
    stillness_pct=0.80,
    label="3min / 80%"
)

## Experiment 4: 3 min window, 70% stillness

In [ ]:
exp4_df, exp4_rem = run_experiment(
    window_sec=180,
    stillness_pct=0.70,
    label="3min / 70%"
)


## Experiment 5: 3 min window, 60% stillness

In [ ]:
exp5_df, exp5_rem = run_experiment(
    window_sec=180,
    stillness_pct=0.60,
    label="3min / 60%"
)
